Fetching the Data

In [1]:
import os
import requests
import math
import pickle
from collections import Counter, defaultdict
import random

Using tinyshakespeare dataset: [dataset](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt)

In [4]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text
# print(text[:250])

Character level tokenization is the first choice.

Frequency counter

In [7]:
def build_ngram_counts(text, n):
  """
  Builds a frequency table.
  Keys: context (string of length n-1)
  Values: Counter object of next-characters
  """
  counts = defaultdict(Counter)
  # If Unigram, context is empty string ''
  if n == 1:
      counts[''] = Counter(text)
      return counts
  # For N > 1, slide a window across the text
  for i in range(len(text) - n + 1):
      context = text[i:i+n-1]
      target = text[i+n-1]
      counts[context][target] += 1

  return counts

In [8]:
bigram_counts = build_ngram_counts(text, n=2)
print("Top 5 characters that follow the letter 't':")
print(bigram_counts['t'].most_common(5))

Top 5 characters that follow the letter 't':
[('h', 22739), (' ', 16508), ('o', 5860), ('e', 4433), ('i', 2977)]


Laplace smoothing

In [9]:
class NGramModel:
    def __init__(self, text, n):
        self.n = n
        # Define the unique vocabulary from the training text
        self.vocab = sorted(list(set(text)))
        self.vocab_size = len(self.vocab)
        self.char_to_idx = {ch: i for i, ch in enumerate(self.vocab)}

        # Build raw counts
        self.counts = build_ngram_counts(text, n)

    def get_probability_distribution(self, context):
        """
        Returns a smoothed probability distribution over the entire vocabulary
        given a context string.
        """
        # Ensure context matches the required length (n-1)
        if self.n > 1:
            # If context is too short, pad it from the left or truncate from the right
            if len(context) < self.n - 1:
                context = context.rjust(self.n - 1)  # simple padding with spaces
            else:
                context = context[-(self.n - 1):]
        else:
            context = ''

        context_counts = self.counts[context]
        total_context_count = sum(context_counts.values())

        probabilities = {}
        # Apply Laplace (Add-1) Smoothing
        for char in self.vocab:
            char_count = context_counts[char]
            smoothed_prob = (char_count + 1) / (total_context_count + self.vocab_size)
            probabilities[char] = smoothed_prob

        return probabilities

# Let's initialize a Bigram model
bigram_model = NGramModel(text, n=2)
print(f"Vocabulary size: {bigram_model.vocab_size} unique characters.")

# Test the distribution for what follows 't'
probs = bigram_model.get_probability_distribution('t')
print("\nSmoothed probabilities for top 5 characters following 't':")
print(sorted(probs.items(), key=lambda x: x[1], reverse=True)[:5])

Vocabulary size: 65 unique characters.

Smoothed probabilities for top 5 characters following 't':
[('h', 0.33902853564719565), (' ', 0.24613113874228465), ('o', 0.0873811014700182), ('e', 0.06610609177922891), ('i', 0.044398723797596684)]


Sampling from the distribution

In [10]:
def generate_text(model, start_context, max_gen_length=200):
    context = start_context
    generated = start_context

    for _ in range(max_gen_length):
        # Get probabilities for the current context
        probs_dict = model.get_probability_distribution(context)

        # Unpack into parallel lists for sampling
        population = list(probs_dict.keys())
        weights = list(probs_dict.values())

        # Sample next character based on weights
        next_char = random.choices(population, weights=weights)[0]

        generated += next_char
        context = generated  # The get_probability_distribution function handles truncation automatically

    return generated

# Try generating from a Bigram model
print("--- Generated Text (Bigram) ---")
print(generate_text(bigram_model, start_context="To be, or not to be"))

--- Generated Text (Bigram) ---
To be, or not to bencontevearuponinest altacofQUERI ge burangin mour, tire wowere
EYow, berser' Issivit ne, s ser I hely t ICou t,
SEDisthe the. make'd,
DYOnd.
me yone hethis,
To owe ys dof ld, sot ly o toverce otinorof


Evaluation

In [11]:
def evaluate_model(model, evaluation_text):
    total_log_prob = 0.0
    count = 0

    # We need to evaluate every character transition in the text
    for i in range(len(evaluation_text)):
        # For the very first characters, context will be shorter than n-1
        if model.n > 1:
            context = evaluation_text[max(0, i - (model.n - 1)):i]
        else:
            context = ''

        target = evaluation_text[i]

        # Get the smoothed probability distribution for this context
        probs = model.get_probability_distribution(context)
        prob = probs[target]

        # Accumulate log probabilities (base 2)
        total_log_prob += math.log2(prob)
        count += 1

    # Calculate cross-entropy and perplexity
    cross_entropy = -total_log_prob / count
    perplexity = 2 ** cross_entropy

    return cross_entropy, perplexity

Split and Validate the model

In [12]:
# 1. Train/Validation Split
split_idx = int(len(text) * 0.9)
train_text = text[:split_idx]
val_text = text[split_idx:]

print(f"Train characters: {len(train_text)}, Validation characters: {len(val_text)}\n")

# 2. Train and Evaluate Unigram (N=1), Bigram (N=2), and Trigram (N=3)
models = {}
for n in [1, 2, 3]:
    name = f"{n}-gram"
    print(f"--- Training {name}... ---")
    model = NGramModel(train_text, n=n)
    models[n] = model

    # Evaluate on validation set
    loss, ppl = evaluate_model(model, val_text)
    print(f"Validation Cross-Entropy: {loss:.4f} bits/char")
    print(f"Validation Perplexity:    {ppl:.2f} (Effective branching factor)")

    # Generate sample text
    sample_prompt = "To be, or not to be" if n > 1 else ""
    generated = generate_text(model, start_context=sample_prompt, max_gen_length=100)
    print(f"Sample Generation:\n\"{generated}\"\n")

Train characters: 1003854, Validation characters: 111540

--- Training 1-gram... ---
Validation Cross-Entropy: 4.8292 bits/char
Validation Perplexity:    28.43 (Effective branching factor)
Sample Generation:
"s uytgeseh  es iheonsa  hsowf  k wyn iwehfe
eouten
,rCygu' h,h  ,,Mhrnr earl earm wert
i atldec
hen!"

--- Training 2-gram... ---
Validation Cross-Entropy: 3.5807 bits/char
Validation Perplexity:    11.96 (Effective branching factor)
Sample Generation:
"To be, or not to be n: cig s fral fugrste, y; d henour.
Thas gn eeede y tikisintth ibe lo d fasive of. he:
Pe pre baran"

--- Training 3-gram... ---
Validation Cross-Entropy: 2.9842 bits/char
Validation Perplexity:    7.91 (Effective branching factor)
Sample Generation:
"To be, or not to be tain o' tworejqG,KWXgjM3.,cHow meried thene.
Me-oplanchis do tho magen we ace thee they frow hichal"



In [13]:
# Save the Bigram (n=2) model
output_filename = "v0_bigram_model.pkl"

with open(output_filename, 'wb') as f:
    pickle.dump(models[2], f)

print(f"Successfully saved {output_filename} ({os.path.getsize(output_filename) / 1024:.2f} KB)")

Successfully saved v0_bigram_model.pkl (7.29 KB)
